In [1]:
import pandas as pd

# Read data from Excel file
excel_file = 'icd10cm.xlsx'  # Update with your file path
df = pd.read_excel(excel_file)


In [2]:
import pandas as pd
import sqlite3
# Connect to SQLite database (or create it)

conn = sqlite3.connect('../database/database.sqlite')


df_transformado = df.rename(columns={
    'Código': 'codigo',
    'Descrição PT_(Longa)': 'descricao_longa',
    'Descrição PT_(Curta)': 'descricao_curta',
    'Capitulo ICD-10-CM_ Código': 'capitulo_codigo',
    'Capitulo ICD-10-CM_desc': 'capitulo_descricao',
    'Capitulo ICD-10-CM_desc_PT': 'capitulo_descricao_pt',
    'Secção ICD-10-CM_Código': 'secao_codigo',
    'Secção ICD-10-CM_Desc': 'secao_descricao',
    'Secção ICD-10-CM_Desc_PT': 'secao_descricao_pt',
    'Válido': 'valido',
    'Ano inicio ': 'ano_inicio',
    'Ano fim': 'ano_fim',
    'Versão': 'versao',
    'Codigo versão anterior': 'codigo_versao_anterior',
    'Tipo Alteração': 'tipo_alteracao'
})

# Selecionar somente as colunas desejadas
colunas_desejadas = [
        'codigo','descricao_longa','descricao_curta','valido','versao','codigo_versao_anterior','tipo_alteracao'
]

df_final = df_transformado[colunas_desejadas]

print(df_final.columns)

# conn.execute('DROP TABLE icd10cms;')
conn.execute('DROP TABLE IF EXISTS icd10cms_new;')

df_final.to_sql('icd10cms', conn, if_exists='replace', index=False)
# add primary key to the table
conn.execute('''
CREATE TABLE icd10cms_new (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo TEXT,
    descricao_longa TEXT,
    descricao_curta TEXT,
    valido TEXT,
    versao TEXT,
    codigo_versao_anterior TEXT,
    tipo_alteracao TEXT
);
''')
conn.execute('''
INSERT INTO icd10cms_new (codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao)
SELECT codigo, descricao_longa, descricao_curta, valido, versao, codigo_versao_anterior, tipo_alteracao
FROM icd10cms;
''')

conn.execute('DROP TABLE icd10cms;')
conn.execute('ALTER TABLE icd10cms_new RENAME TO icd10cms;')

conn.commit()

# Close the database connection
print("Database connection closed.")

conn.close()

Index(['codigo', 'descricao_longa', 'descricao_curta', 'valido', 'versao',
       'codigo_versao_anterior', 'tipo_alteracao'],
      dtype='object')
Database connection closed.
